# Прогноз интенсивности автомобильного движения
Лабораторная работа №1: синхронный инференс регрессионной модели.
Источник: [Metro Interstate Traffic Volume, UCI](https://doi.org/10.24432/C5X60B),
John Hogue (2019), CC BY 4.0. Цель — число автомобилей за час на посту
ATR 301, западное направление I-94 между Minneapolis и St Paul.
Используется 10 000 наблюдений и 7 признаков.
Имя notebook сохранено из исходного CloudCS-Lab1; содержимое относится к трафику.

In [1]:
import json
import platform
from pathlib import Path
from pickle import dump, load
from time import perf_counter

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
raw = pd.read_csv(ROOT / "data/Metro_Interstate_Traffic_Volume.csv.gz",
                  keep_default_na=False, na_values=[""])
raw["date_time"] = pd.to_datetime(raw["date_time"], errors="raise")
raw.head()

,holiday,temp,rain_1h,snow_1h,clouds_all,weather_main,weather_description,date_time,traffic_volume
0,None,288.28,0.0,0.0,40,Clouds,scattered clouds,2012-10-02 09:00:00,5545
1,None,289.36,0.0,0.0,75,Clouds,broken clouds,2012-10-02 10:00:00,4516
2,None,289.58,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 11:00:00,4767
3,None,290.13,0.0,0.0,90,Clouds,overcast clouds,2012-10-02 12:00:00,5026
4,None,291.14,0.0,0.0,75,Clouds,broken clouds,2012-10-02 13:00:00,4918


In [2]:
raw.shape

(48204, 9)

In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48204 entries, 0 to 48203
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   holiday              48204 non-null  object        
 1   temp                 48204 non-null  float64       
 2   rain_1h              48204 non-null  float64       
 3   snow_1h              48204 non-null  float64       
 4   clouds_all           48204 non-null  int64         
 5   weather_main         48204 non-null  object        
 6   weather_description  48204 non-null  object        
 7   date_time            48204 non-null  datetime64[ns]
 8   traffic_volume       48204 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(2), object(3)
memory usage: 3.3+ MB


In [4]:
raw.describe()

,temp,rain_1h,snow_1h,clouds_all,date_time,traffic_volume
count,48204.000000,48204.000000,48204.000000,48204.000000,48204,48204.000000
mean,281.205870,0.334264,0.000222,49.362231,2016-01-05 10:46:16.773711616,3259.818355
min,0.000000,0.000000,0.000000,0.000000,2012-10-02 09:00:00,0.000000
25%,272.160000,0.000000,0.000000,1.000000,2014-02-06 11:45:00,1193.000000
50%,282.450000,0.000000,0.000000,64.000000,2016-06-11 03:30:00,3380.000000
75%,291.806000,0.000000,0.000000,90.000000,2017-08-11 06:00:00,4933.000000
max,310.070000,9831.300000,0.510000,100.000000,2018-09-30 23:00:00,7280.000000
std,13.338232,44.789133,0.008168,39.015750,NaN,1986.860670


In [5]:
display(raw.isna().sum())
print("Дубликаты исходных строк:", int(raw.duplicated().sum()))
print("Повторные временные отметки:", int(raw.date_time.duplicated().sum()))
print("Температура <= 0 K:", int((raw.temp <= 0).sum()))
print("Дождь > 1000 мм за час:", int((raw.rain_1h > 1000).sum()))

holiday                0
temp                   0
rain_1h                0
snow_1h                0
clouds_all             0
weather_main           0
weather_description    0
date_time              0
traffic_volume         0
dtype: int64

Дубликаты исходных строк: 17
Повторные временные отметки: 7629
Температура <= 0 K: 10
Дождь > 1000 мм за час: 1


## Подготовка 10 000 наблюдений
В источнике одному часу могут соответствовать несколько описаний погоды.
После сортировки оставляется первая исходная строка каждого часа: решение
не зависит от целевой переменной. Нулевая температура в кельвинах и дождь
свыше 1000 мм за час исключаются как явные некорректные измерения.
Это фиксированные правила качества данных, а не подбор по метрикам модели.
Строка `None` в holiday означает отсутствие праздника, а не пропуск.
Остальные исходные столбцы не входят в модель.

Из очищенных уникальных часов выбираются 10 000 строк с random_state=42;
затем они сортируются по времени. Это выборка по всему периоду, а не
10 000 последовательных часов. Дата сохраняется для аудита разделения,
но не передаётся в модель.

In [6]:
invalid = (raw.temp <= 0) | (raw.rain_1h > 1000)
clean = raw.loc[~invalid].sort_values("date_time", kind="stable")
clean = clean.drop_duplicates("date_time", keep="first")
sample = clean.sample(n=10000, random_state=42).sort_values("date_time").copy()
weekdays = dict(enumerate(["Monday", "Tuesday", "Wednesday", "Thursday",
                           "Friday", "Saturday", "Sunday"]))
months = dict(enumerate(["January", "February", "March", "April", "May", "June",
                        "July", "August", "September", "October", "November",
                        "December"], start=1))
df = pd.DataFrame({
    "date_time": sample.date_time,
    "hour": sample.date_time.dt.hour,
    "day_of_week": sample.date_time.dt.dayofweek.map(weekdays),
    "month": sample.date_time.dt.month.map(months),
    "weather_main": sample.weather_main,
    "temperature_c": sample.temp - 273.15,
    "rain_1h": sample.rain_1h,
    "clouds_all": sample.clouds_all,
    "traffic_volume": sample.traffic_volume,
}).reset_index(drop=True)
df.to_csv(ROOT / "data/traffic_10000.csv", index=False)
df.head()

,date_time,hour,day_of_week,month,weather_main,temperature_c,rain_1h,clouds_all,traffic_volume
0,2012-10-02 10:00:00,10,Tuesday,October,Clouds,16.21,0.0,75,4516
1,2012-10-02 13:00:00,13,Tuesday,October,Clouds,17.99,0.0,75,4918
2,2012-10-02 15:00:00,15,Tuesday,October,Clear,20.02,0.0,1,5584
3,2012-10-02 16:00:00,16,Tuesday,October,Clear,20.71,0.0,1,6015
4,2012-10-02 22:00:00,22,Tuesday,October,Clear,14.01,0.0,1,1529


In [7]:
df.shape

(10000, 9)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date_time       10000 non-null  datetime64[ns]
 1   hour            10000 non-null  int32         
 2   day_of_week     10000 non-null  object        
 3   month           10000 non-null  object        
 4   weather_main    10000 non-null  object        
 5   temperature_c   10000 non-null  float64       
 6   rain_1h         10000 non-null  float64       
 7   clouds_all      10000 non-null  int64         
 8   traffic_volume  10000 non-null  int64         
dtypes: datetime64[ns](1), float64(2), int32(1), int64(2), object(3)
memory usage: 664.2+ KB


In [9]:
df.describe()

,date_time,hour,temperature_c,rain_1h,clouds_all,traffic_volume
count,10000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000
mean,2015-12-20 04:38:54.600000,11.57490,8.368233,0.082273,44.109200,3307.650400
min,2012-10-02 10:00:00,0.00000,-28.330000,0.000000,0.000000,0.000000
25%,2014-01-25 21:00:00,6.00000,-1.180000,0.000000,1.000000,1289.000000
50%,2016-06-01 18:00:00,12.00000,10.000000,0.000000,40.000000,3448.000000
75%,2017-08-02 23:30:00,18.00000,19.195500,0.000000,90.000000,4969.250000
max,2018-09-30 18:00:00,23.00000,35.800000,19.900000,100.000000,7260.000000
std,NaN,6.95257,13.017046,0.685419,38.702993,1978.948511


In [10]:
missing = df.isna().sum()
display(missing)
assert missing.sum() == 0, "При появлении пропусков потребуется imputer внутри Pipeline"
print("Пропусков нет, заполнение не требуется.")
print("Дубликаты полных подготовленных строк:", int(df.duplicated().sum()))
print("Совпадения признаков и цели без даты:",
      int(df.drop(columns="date_time").duplicated().sum()))

date_time         0
hour              0
day_of_week       0
month             0
weather_main      0
temperature_c     0
rain_1h           0
clouds_all        0
traffic_volume    0
dtype: int64

Пропусков нет, заполнение не требуется.
Дубликаты полных подготовленных строк: 0
Совпадения признаков и цели без даты: 0


In [11]:
categorical_features = ["day_of_week", "month", "weather_main"]
numeric_features = ["hour", "temperature_c", "rain_1h", "clouds_all"]
features = ["hour", "day_of_week", "month", "weather_main",
            "temperature_c", "rain_1h", "clouds_all"]
display(df[categorical_features].nunique())
for column in categorical_features:
    print(column)
    display(df[column].value_counts())
categories = {column: sorted(df[column].unique().tolist())
              for column in categorical_features}
(ROOT / "data/categories.json").write_text(
    json.dumps(categories, indent=2), encoding="utf-8")

day_of_week      7
month           12
weather_main    10
dtype: int64

day_of_week


day_of_week
Friday       1467
Tuesday      1449
Sunday       1439
Monday       1435
Thursday     1412
Saturday     1402
Wednesday    1396
Name: count, dtype: int64

month


month
July         1033
August        933
May           911
June          835
April         810
March         801
September     795
December      793
February      782
January       775
November      774
October       758
Name: count, dtype: int64

weather_main


weather_main
Clouds          3670
Clear           3335
Rain            1212
Mist             740
Snow             570
Haze             185
Drizzle          120
Thunderstorm     114
Fog               50
Smoke              4
Name: count, dtype: int64

476

## Разделение по времени
Первые 8000 строк используются для обучения, последние 2000 — для проверки.
Ни одна временная отметка не попадает одновременно в обе части.
Модель оценивается на более позднем периоде.
Приведённые метрики относятся к проверочной выборке; независимое подтверждение качества требует новых данных.

In [12]:
X = df[features]
y = df["traffic_volume"]
X_train, X_test = X.iloc[:8000], X.iloc[8000:]
y_train, y_test = y.iloc[:8000], y.iloc[8000:]
assert df.date_time.is_unique
assert df.date_time.iloc[:8000].max() < df.date_time.iloc[8000:].min()
print("Обучение:", X_train.shape, df.date_time.iloc[0], df.date_time.iloc[7999])
print("Тест:", X_test.shape, df.date_time.iloc[8000], df.date_time.iloc[-1])

Обучение: (8000, 7) 2012-10-02 10:00:00 2017-10-22 14:00:00
Тест: (2000, 7) 2017-10-22 16:00:00 2018-09-30 18:00:00


## Pipeline
OneHotEncoder преобразует три строковых признака в бинарные столбцы.
ColumnTransformer объединяет категории и четыре числовых признака.
Числа передаются без масштабирования: деревьям StandardScaler не требуется.
SimpleImputer не добавляется, так как проверка не обнаружила пропусков.
RandomForestRegressor усредняет прогнозы 200 деревьев на bootstrap-выборках.
n_jobs=4 ограничивает параллельное обучение четырьмя рабочими потоками.

In [13]:
preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("numeric", "passthrough", numeric_features),
])
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=4)),
])
started = perf_counter()
pipeline.fit(X_train, y_train)
training_seconds = perf_counter() - started
print(f"Время обучения в текущей среде: {training_seconds:.2f} с")

Время обучения в текущей среде: 23.57 с


In [14]:
prediction = pipeline.predict(X_test)
metrics = {
    "MAE": float(mean_absolute_error(y_test, prediction)),
    "RMSE": float(np.sqrt(mean_squared_error(y_test, prediction))),
    "R2": float(r2_score(y_test, prediction)),
}
for name, value in metrics.items():
    print(f"{name}: {value:.6f}")

MAE: 298.067725
RMSE: 538.952747
R2: 0.926432


Ошибка MAE измеряется в автомобилях за час. Отношение MAE к среднему трафику
даёт представление о масштабе ошибки, но не является MAPE или «точностью».
Нулевая ошибка не ожидается: семь признаков не описывают аварии, перекрытия
и все особенности праздничных дней. Для применения к будущему часу нужны
прогнозные погодные данные; тест использует наблюдавшуюся погоду.
Метрики относятся к одному посту учёта, а не к произвольной дороге.

In [15]:
(ROOT / "models").mkdir(exist_ok=True)
with (ROOT / "models/pipeline.pkl").open("wb") as file:
    dump(pipeline, file)
with (ROOT / "models/pipeline.pkl").open("rb") as file:
    restored = load(file)
np.testing.assert_allclose(restored.predict(X_test), prediction)
example = json.loads((ROOT / "data/example_request.json").read_text())
example_prediction = float(restored.predict(pd.DataFrame([example]))[0])
print({"predicted_traffic_volume": round(example_prediction, 3)})
unknown = {**example, "weather_main": "Unknown weather"}
assert np.isfinite(restored.predict(pd.DataFrame([unknown]))).all()

{'predicted_traffic_volume': 4265.89}


In [16]:
report = {
    "source_rows": len(raw), "source_missing": int(raw.isna().sum().sum()),
    "source_duplicates": int(raw.duplicated().sum()),
    "invalid_measurements_removed": int(invalid.sum()),
    "duplicate_hours_removed": int((~invalid).sum() - len(clean)),
    "eligible_unique_hours": len(clean), "rows": len(df),
    "features": features, "missing": int(missing.sum()),
    "duplicates": int(df.duplicated().sum()),
    "duplicates_without_date": int(df.drop(columns="date_time").duplicated().sum()),
    "train_rows": len(X_train), "test_rows": len(X_test),
    "train_start": str(df.date_time.iloc[0]), "train_end": str(df.date_time.iloc[7999]),
    "test_start": str(df.date_time.iloc[8000]), "test_end": str(df.date_time.iloc[-1]),
    "metrics": metrics,
    "test_target_mean": float(y_test.mean()),
    "test_target_median": float(y_test.median()),
    "parameters": pipeline["model"].get_params(),
    "training_seconds": training_seconds, "example_prediction": example_prediction,
    "python": platform.python_version(), "scikit_learn": sklearn.__version__,
}
(ROOT / "models/metrics.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8")
display(report)

{'source_rows': 48204,
 'source_missing': 0,
 'source_duplicates': 17,
 'invalid_measurements_removed': 11,
 'duplicate_hours_removed': 7629,
 'eligible_unique_hours': 40564,
 'rows': 10000,
 'features': ['hour',
  'day_of_week',
  'month',
  'weather_main',
  'temperature_c',
  'rain_1h',
  'clouds_all'],
 'missing': 0,
 'duplicates': 0,
 'duplicates_without_date': 0,
 'train_rows': 8000,
 'test_rows': 2000,
 'train_start': '2012-10-02 10:00:00',
 'train_end': '2017-10-22 14:00:00',
 'test_start': '2017-10-22 16:00:00',
 'test_end': '2018-09-30 18:00:00',
 'metrics': {'MAE': 298.0677253333333,
  'RMSE': 538.9527467642595,
  'R2': 0.9264321470091116},
 'test_target_mean': 3330.7545,
 'test_target_median': 3488.0,
 'parameters': {'bootstrap': True,
  'ccp_alpha': 0.0,
  'criterion': 'squared_error',
  'max_depth': None,
  'max_features': 1.0,
  'max_leaf_nodes': None,
  'max_samples': None,
  'min_impurity_decrease': 0.0,
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'min_weight_